# 07 · Build a project-specific verb

Domain add-ons are usually projects that define their own verbs. This notebook
implements a small heat-stress vocabulary without modifying or registering
anything inside CubeDynamics, verifies direct and pipe use, and plots the
derived state and magnitude.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe


def heat_stress(*, threshold: float = 35.0):
    '''Return a project-owned cube → Dataset verb.'''
    def _op(cube: xr.DataArray) -> xr.Dataset:
        if "time" not in cube.dims:
            raise ValueError("heat_stress requires a 'time' dimension")
        state = (cube >= threshold).rename("state")
        magnitude = (cube - threshold).where(state, 0).rename("magnitude")
        result = xr.Dataset({"state": state, "magnitude": magnitude})
        result.attrs.update(cube.attrs)
        result.attrs.update(project_verb="heat_stress", threshold=float(threshold))
        return result
    return _op


time = pd.date_range("2025-07-01", periods=10, freq="D")
y = [1, 0]
x = [0, 1, 2]
pulse = np.array([0, 1, 3, 6, 8, 5, 2, 0, -1, 1])[:, None, None]
spatial = np.array([[-1.0, 0.0, 1.0], [0.0, 1.0, 2.0]])[None, :, :]
temperature = xr.DataArray(
    31 + pulse + spatial,
    dims=("time", "y", "x"),
    coords={"time": time, "y": y, "x": x},
    name="air_temperature",
    attrs={"units": "degC", "source": "deterministic synthetic vignette"},
)

direct = heat_stress(threshold=35.0)(temperature)
through_pipe = (pipe(temperature) | heat_stress(threshold=35.0)).unwrap()
xr.testing.assert_identical(direct, through_pipe)
daily_fraction = through_pipe["state"].mean(("y", "x"))
cumulative_magnitude = through_pipe["magnitude"].sum("time")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
temperature.mean(("y", "x")).plot(ax=axes[0], marker="o", color="#8b543c")
axes[0].axhline(35, color="0.35", linestyle="--")
axes[0].set_title("Input and project threshold")
daily_fraction.plot(ax=axes[1], marker="o", color="#3f6f72")
axes[1].set_title("Derived heat-stress fraction")
axes[1].set_ylim(-0.05, 1.05)
cumulative_magnitude.plot(ax=axes[2], cmap="YlOrRd", cbar_kwargs={"label": "degree-days"})
axes[2].set_title("Derived cumulative magnitude")
plt.show()

In a real add-on, move `heat_stress` into `my_project.verbs`, document the
threshold's scientific meaning, and keep the direct-versus-pipe regression test
with that project. The repository's `examples/custom_verb_project/` directory
provides a minimal package layout.